# Detector personalizado de objetos con YOLO26 y Roboflow

**Objetivo.** Entrenar un modelo YOLO que reconozca tres objetos propios que el modelo preentrenado no conoce:
un **termo**, un **mando de videojuegos** y una **plancha**, y usarlo en tiempo real con la webcam.

**Proceso.**

1. Grabar un video de cada objeto y extraer cuadros nítidos y variados (`src/extract_frames.py`).
2. Subir las fotos a Roboflow, anotarlas con cajas y generar una versión del dataset.
3. Descargar el dataset y entrenar YOLO26 en este notebook.
4. Evaluar el modelo con las métricas estándar de detección.
5. Probarlo sobre imágenes nuevas y en tiempo real con la webcam (`src/detect_webcam.py`).

## 1. Configuración

In [ ]:
import collections
import json
import os
import shutil
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import yaml
from dotenv import load_dotenv
from roboflow import Roboflow
from ultralytics import YOLO

# Raíz del proyecto (este notebook vive en la carpeta notebooks/)
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

FOTOS_DIR = ROOT / 'data' / 'fotos'          # fotos originales extraídas de los videos
DATASET_DIR = ROOT / 'datasets' / 'objetos-personales'   # dataset descargado de Roboflow
RUNS_DIR = ROOT / 'runs'                     # resultados del entrenamiento
REPORTS_DIR = ROOT / 'reports'               # gráficas y resumen para el informe
REPORTS_DIR.mkdir(exist_ok=True)

# Hiperparámetros del entrenamiento
BASE_MODEL = 'yolo26n.pt'   # YOLO26 versión nano: liviano y suficiente para un dataset pequeño
EPOCHS = 50                 # pasadas completas por el dataset
IMGSZ = 640                 # tamaño de las imágenes de entrenamiento
BATCH = 8                   # imágenes por lote
PATIENCE = 15               # se detiene si no mejora durante 15 épocas
RUN_NAME = 'custom_yolo_model'

## 2. Dataset

Las fotos salen de videos grabados con el celular. El script `src/extract_frames.py` descarta los cuadros borrosos y
los que se parecen demasiado entre sí, y guarda los más variados. Hay una carpeta por tipo de foto:

- `termo`, `mando` y `plancha`: cada objeto solo, desde distintos ángulos, distancias, fondos y luces.
- `juntos`: los tres objetos en la misma escena.
- `fondo`: escenas sin ninguno de los objetos, que quedan sin cajas para que el modelo aprenda lo que no es un objeto.

In [ ]:
# Fotos originales por carpeta
total = 0
print(f"{'Carpeta':<10}{'Fotos':>6}")
for carpeta in sorted(p for p in FOTOS_DIR.iterdir() if p.is_dir()):
    n = len(list(carpeta.glob('*.jpg')))
    total += n
    print(f'{carpeta.name:<10}{n:>6}')
print(f"{'TOTAL':<10}{total:>6}")

### 2.1 Descarga desde Roboflow

El dataset anotado se descarga con la API de Roboflow. La clave y los datos del proyecto se leen del archivo `.env`,
que no se sube a GitHub. Si el dataset ya está en `datasets/`, se reutiliza y no se vuelve a descargar.

In [ ]:
load_dotenv(ROOT / '.env')

if not (DATASET_DIR / 'data.yaml').exists():
    api_key = os.getenv('ROBOFLOW_API_KEY')
    if not api_key:
        raise RuntimeError('Falta ROBOFLOW_API_KEY. Copia .env.example como .env y completa tus datos.')
    rf = Roboflow(api_key=api_key)
    project = rf.workspace(os.getenv('ROBOFLOW_WORKSPACE')).project(os.getenv('ROBOFLOW_PROJECT'))
    version = project.version(int(os.getenv('ROBOFLOW_VERSION', '1')))
    try:
        version.download('yolo26', location=str(DATASET_DIR))
    except Exception:
        version.download('yolov8', location=str(DATASET_DIR))   # mismo formato de etiquetas YOLO
else:
    print('Dataset ya descargado en', DATASET_DIR)

# Nombres de las clases según el data.yaml de Roboflow
with open(DATASET_DIR / 'data.yaml') as f:
    roboflow_cfg = yaml.safe_load(f)
names = roboflow_cfg['names']
NAMES = dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k, v in names.items()}
print('Clases:', NAMES)

# data.yaml con rutas absolutas, para que Ultralytics encuentre siempre las carpetas
data_cfg = {'path': str(DATASET_DIR), 'train': 'train/images', 'val': 'valid/images', 'names': NAMES}
if (DATASET_DIR / 'test' / 'images').exists():
    data_cfg['test'] = 'test/images'
DATA_YAML = DATASET_DIR / 'data_local.yaml'
with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
EVAL_SPLIT = 'test' if 'test' in data_cfg else 'val'
print('Evaluación final sobre el conjunto:', EVAL_SPLIT)

### 2.2 Resumen del dataset

Imágenes y cajas (instancias) por clase en cada conjunto. Las clases deben estar equilibradas para que el modelo no
favorezca a ninguna.

In [ ]:
def contar(split):
    """Cuenta las imágenes de un conjunto y las cajas por clase a partir de sus etiquetas YOLO."""
    img_dir, lbl_dir = DATASET_DIR / split / 'images', DATASET_DIR / split / 'labels'
    if not img_dir.exists():
        return None
    n_img = len([p for p in img_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
    inst = collections.Counter()
    for label_file in lbl_dir.glob('*.txt'):
        for line in label_file.read_text().splitlines():
            if line.strip():
                inst[NAMES[int(line.split()[0])]] += 1
    return n_img, inst

resumen_dataset = {}
print(f"{'Conjunto':<10}{'Imágenes':>9}   Cajas por clase")
for split in ('train', 'valid', 'test'):
    c = contar(split)
    if c is None:
        continue
    resumen_dataset[split] = {'imagenes': c[0], 'cajas': dict(c[1])}
    print(f'{split:<10}{c[0]:>9}   {dict(c[1])}')

# Gráfica de cajas por clase y conjunto
fig, ax = plt.subplots(figsize=(8, 4))
splits = list(resumen_dataset)
ancho = 0.8 / max(len(splits), 1)
for i, split in enumerate(splits):
    valores = [resumen_dataset[split]['cajas'].get(NAMES[k], 0) for k in sorted(NAMES)]
    ax.bar([k + i * ancho for k in range(len(NAMES))], valores, width=ancho, label=split)
ax.set_xticks([k + ancho * (len(splits) - 1) / 2 for k in range(len(NAMES))])
ax.set_xticklabels([NAMES[k] for k in sorted(NAMES)])
ax.set_ylabel('Cajas anotadas')
ax.set_title('Cajas por clase y conjunto')
ax.legend()
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'dataset_clases.png', dpi=110)
plt.show()

## 3. Entrenamiento

Se parte de **YOLO26 nano preentrenado en COCO** y se ajusta (*fine-tuning*) con el dataset propio. Aprovechar los pesos
preentrenados permite lograr un buen modelo con pocas imágenes.

| Parámetro | Valor | Por qué |
|---|---|---|
| Modelo base | `yolo26n.pt` | Liviano; suficiente para 3 clases y rápido en CPU |
| Épocas | 50 | Con parada anticipada si no hay mejora (`patience`) |
| Tamaño de imagen | 640 | Tamaño estándar de YOLO |
| Lote | 8 | Cabe en la memoria de un equipo sin GPU |

In [ ]:
model = YOLO(BASE_MODEL)

results = model.train(
    data=str(DATA_YAML),        # dataset en formato YOLO
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    project=str(RUNS_DIR),      # carpeta de resultados
    name=RUN_NAME,
    exist_ok=True,              # sobrescribe el entrenamiento anterior con el mismo nombre
    workers=0,                  # evita problemas de multiproceso en Windows
    plots=True,                 # guarda las curvas y la matriz de confusión
)
BEST_MODEL = RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'
print('Mejor modelo guardado en:', BEST_MODEL)

### 3.1 Curvas de entrenamiento

La pérdida (*loss*) debe bajar y las métricas de validación deben subir hasta estabilizarse.

In [ ]:
curvas = RUNS_DIR / RUN_NAME / 'results.png'
img = plt.imread(curvas)
plt.figure(figsize=(14, 7))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()
shutil.copy(curvas, REPORTS_DIR / 'curvas_entrenamiento.png')

## 4. Evaluación

Se evalúa el mejor modelo sobre imágenes que **no se usaron para entrenar**. Las métricas:

- **Precisión (P):** de lo que el modelo detecta, qué fracción es correcta.
- **Recall (R):** de los objetos que hay, qué fracción detecta.
- **mAP50:** precisión media con un solapamiento mínimo (IoU) de 0.5 entre la caja predicha y la real.
- **mAP50-95:** promedio con solapamientos de 0.5 a 0.95; es la métrica más exigente.

In [ ]:
best = YOLO(str(BEST_MODEL))
metrics = best.val(data=str(DATA_YAML), split=EVAL_SPLIT, imgsz=IMGSZ, workers=0,
                   project=str(RUNS_DIR), name='evaluacion', exist_ok=True, plots=True)

print()
print(f"{'':<10}{'P':>8}{'R':>8}{'mAP50':>9}{'mAP50-95':>10}")
print(f"{'GLOBAL':<10}{metrics.box.mp:>8.3f}{metrics.box.mr:>8.3f}{metrics.box.map50:>9.3f}{metrics.box.map:>10.3f}")
por_clase = {}
for i, cls_idx in enumerate(metrics.box.ap_class_index):
    p, r, ap50, ap = metrics.box.class_result(i)
    por_clase[NAMES[int(cls_idx)]] = {'P': float(p), 'R': float(r), 'mAP50': float(ap50), 'mAP50-95': float(ap)}
    print(f'{NAMES[int(cls_idx)]:<10}{p:>8.3f}{r:>8.3f}{ap50:>9.3f}{ap:>10.3f}')

### 4.1 Matriz de confusión y curva precisión-recall

In [ ]:
eval_dir = RUNS_DIR / 'evaluacion'
graficas = [('confusion_matrix_normalized.png', 'matriz_confusion.png'),
            ('BoxPR_curve.png', 'curva_precision_recall.png')]
disponibles = [(a, b) for a, b in graficas if (eval_dir / a).exists()]
fig, axes = plt.subplots(1, len(disponibles), figsize=(7 * len(disponibles), 6))
axes = [axes] if len(disponibles) == 1 else axes
for ax, (origen, destino) in zip(axes, disponibles):
    ax.imshow(plt.imread(eval_dir / origen))
    ax.axis('off')
    shutil.copy(eval_dir / origen, REPORTS_DIR / destino)
plt.tight_layout()
plt.show()

### 4.2 Análisis de los resultados

El texto se genera a partir de las métricas obtenidas, para que siempre coincida con ellas.

In [ ]:
ordenadas = sorted(por_clase.items(), key=lambda kv: kv[1]['mAP50-95'], reverse=True)
mejor, peor = ordenadas[0], ordenadas[-1]
print(f"Conjunto de evaluación: {EVAL_SPLIT}")
print(f"El modelo alcanza un mAP50 de {metrics.box.map50:.3f} y un mAP50-95 de {metrics.box.map:.3f}.")
print(f"La clase mejor detectada es '{mejor[0]}' (mAP50-95 = {mejor[1]['mAP50-95']:.3f}).")
print(f"La clase más difícil es '{peor[0]}' (mAP50-95 = {peor[1]['mAP50-95']:.3f}, "
      f"precisión {peor[1]['P']:.3f}, recall {peor[1]['R']:.3f}).")
if peor[1]['R'] < peor[1]['P']:
    print(f"En '{peor[0]}' el modelo es preciso pero se le escapan objetos: conviene agregar más fotos con "
          "ese objeto parcialmente tapado, lejano o con poca luz.")
else:
    print(f"En '{peor[0]}' el modelo detecta los objetos pero confunde algunos con otras clases o con el fondo: "
          "conviene revisar sus anotaciones y agregar ejemplos negativos.")

**Cómo interpretar las métricas.** Las fotos de entrenamiento y de evaluación salen de los mismos videos, por lo que se
parecen entre sí y las métricas tienden a ser altas. La prueba más exigente es la detección en tiempo real con la
webcam (sección 6), porque trabaja con imágenes completamente nuevas.

## 5. Prueba sobre imágenes nuevas

Detecciones del mejor modelo sobre imágenes del conjunto de evaluación.

In [ ]:
test_dir = DATASET_DIR / ('test' if EVAL_SPLIT == 'test' else 'valid') / 'images'
muestra = sorted(test_dir.glob('*.jpg'))[:8]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, path in zip(axes.ravel(), muestra):
    res = best(str(path), conf=0.4, imgsz=IMGSZ, verbose=False)[0]
    ax.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB))   # res.plot() devuelve BGR; matplotlib usa RGB
    ax.axis('off')
for ax in axes.ravel()[len(muestra):]:
    ax.axis('off')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'pruebas_imagenes.png', dpi=100)
plt.show()

## 6. Detección en tiempo real con la webcam

La función `run` de `src/detect_webcam.py` abre la webcam, detecta en cada cuadro con el modelo entrenado y dibuja las
cajas con su clase y confianza, un contador por clase y los cuadros por segundo.

- **Q** cierra la ventana.
- **S** guarda una captura en `reports/`.

La ventana de video solo se abre al ejecutar el notebook en **Windows**, con la webcam conectada.

In [ ]:
from src.detect_webcam import run

# Muestra la webcam con las detecciones hasta que se pulse Q
run(modelo=BEST_MODEL, camara=0, conf=0.5)

## 7. Resumen

Se guarda un resumen con el dataset y las métricas del modelo en `reports/resumen.json`.

In [ ]:
resumen = {
    'clases': list(NAMES.values()),
    'dataset': resumen_dataset,
    'entrenamiento': {'modelo_base': BASE_MODEL, 'epocas': EPOCHS, 'imgsz': IMGSZ, 'batch': BATCH},
    'evaluacion': {
        'conjunto': EVAL_SPLIT,
        'global': {'P': float(metrics.box.mp), 'R': float(metrics.box.mr),
                   'mAP50': float(metrics.box.map50), 'mAP50-95': float(metrics.box.map)},
        'por_clase': por_clase,
    },
}
with open(REPORTS_DIR / 'resumen.json', 'w', encoding='utf-8') as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)
print(json.dumps(resumen, indent=2, ensure_ascii=False))